#### Notebook to explore `ai4privacy/pii-masking-300k` dataset

In [1]:
import os
from pathlib import Path
import json
from pprint import pprint
from collections import Counter

from colorama import Fore, Style

from datasets import load_dataset
from huggingface_hub import dataset_info

from tqdm import tqdm

In [2]:
ds_path = "ai4privacy/pii-masking-300k"
ds = load_dataset(ds_path)

In [3]:
# dataset card data
info = dataset_info(ds_path)
print(info.card_data)

language:
- en
- fr
- de
- it
- es
- nl
license: other
multilinguality:
- multilingual
size_categories:
- 100K<n<1M
source_datasets:
- original
task_categories:
- text-classification
- token-classification
- table-question-answering
- question-answering
- zero-shot-classification
- summarization
- feature-extraction
- text-generation
- translation
- fill-mask
- tabular-classification
- tabular-to-text
- table-to-text
- text-retrieval
- other
pretty_name: Ai4Privacy PII 300k Dataset
license_name: license.md
tags:
- legal
- business
- psychology
- privacy
- gdpr
- euaiact
- aiact
- pii
- sensitive
configs:
- config_name: default
  data_files:
  - split: train
    path: data/train/*.jsonl
  - split: validation
    path: data/validation/*.jsonl


In [4]:
# get dataset repr
print(repr(ds))
print("=" * 100, "\n")

DatasetDict({
    train: Dataset({
        features: ['source_text', 'target_text', 'privacy_mask', 'span_labels', 'mbert_text_tokens', 'mbert_bio_labels', 'id', 'language', 'set'],
        num_rows: 177677
    })
    validation: Dataset({
        features: ['source_text', 'target_text', 'privacy_mask', 'span_labels', 'mbert_text_tokens', 'mbert_bio_labels', 'id', 'language', 'set'],
        num_rows: 47728
    })
})



In [5]:
ds_train = ds["train"]
# get ds features
pprint(ds_train.features)

{'id': Value('string'),
 'language': Value('string'),
 'mbert_bio_labels': List(Value('string')),
 'mbert_text_tokens': List(Value('string')),
 'privacy_mask': List({'value': Value('string'), 'start': Value('int64'), 'end': Value('int64'), 'label': Value('string')}),
 'set': Value('string'),
 'source_text': Value('string'),
 'span_labels': Value('string'),
 'target_text': Value('string')}


In [6]:
# this dataset is multilingual. i will train on just the english examples for simplicity
print(f"length of full dataset: {len(ds_train):,}")
ds_train = ds_train.filter(lambda x: x["language"] == "English")
print(f"English only ds length: {len(ds_train):,}")

length of full dataset: 177,677


Filter:   0%|          | 0/177677 [00:00<?, ? examples/s]

English only ds length: 29,908


In [7]:
# view single example from dataset
example = ds_train[0]

for i in example:
    print(Fore.CYAN + f"{i}:")
    print(Style.RESET_ALL)
    pprint(f"{example[i]}")
    print("\n\n")
    

source_text:

('Subject: Group Messaging for Admissions Process\n'
 '\n'
 'Good morning, everyone,\n'
 '\n'
 'I hope this message finds you well. As we continue our admissions processes, '
 'I would like to update you on the latest developments and key information. '
 'Please find below the timeline for our upcoming meetings:\n'
 '\n'
 '- wynqvrh053 - Meeting at 10:20am\n'
 '- luka.burg - Meeting at 21\n'
 '- qahil.wittauer - Meeting at quarter past 13\n'
 '- gholamhossein.ruschke - Meeting at 9:47 PM\n'
 '- pdmjrsyoz1460 ')



target_text:

('Subject: Group Messaging for Admissions Process\n'
 '\n'
 'Good morning, everyone,\n'
 '\n'
 'I hope this message finds you well. As we continue our admissions processes, '
 'I would like to update you on the latest developments and key information. '
 'Please find below the timeline for our upcoming meetings:\n'
 '\n'
 '- [USERNAME] - Meeting at [TIME]\n'
 '- [USERNAME] - Meeting at [TIME]\n'
 '- [USERNAME] - Meeting at [TIME]\n'
 '- [USERNAME] 

In [8]:
print(len(example["mbert_text_tokens"]))
print(len(example["mbert_bio_labels"]))

121
121


There isn't a feature in this dataset to get the number of labels and their names.

In [9]:
# get the labels feature from the dataset
labels_feat = ds_train["mbert_bio_labels"] 
print(next(iter(labels_feat)))

['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-USERNAME', 'I-USERNAME', 'I-USERNAME', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-TIME', 'I-TIME', 'I-TIME', 'O', 'B-USERNAME', 'I-USERNAME', 'O', 'O', 'O', 'B-TIME', 'I-TIME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'O', 'O', 'O', 'O', 'B-TIME', 'I-TIME', 'I-TIME', 'I-TIME', 'I-TIME', 'I-TIME', 'I-TIME', 'I-TIME', 'I-TIME', 'I-TIME', 'O', 'B-USERNAME', 'I-USERNAME']


In [10]:
# count every label.
bio_counter = Counter()
with tqdm(labels_feat) as pbar:
    for label in pbar:
        bio_counter.update(label)

100%|██████████| 29908/29908 [00:02<00:00, 12959.86it/s]


In [11]:
print(f"num_labels = {len(bio_counter.items())}")
    
print(Fore.CYAN + "sorted labels count for train_dataset:")
print(Style.RESET_ALL, end="")
for i in sorted(bio_counter.items()):
    print(i)
    

num_labels = 57
sorted labels count for train_dataset:
('B-BOD', 7740)
('B-BUILDING', 6062)
('B-CARDISSUER', 5)
('B-CITY', 5964)
('B-COUNTRY', 4844)
('B-DATE', 5308)
('B-DRIVERLICENSE', 7967)
('B-EMAIL', 9124)
('B-GEOCOORD', 535)
('B-GIVENNAME1', 6836)
('B-GIVENNAME2', 1589)
('B-IDCARD', 8667)
('B-IP', 7692)
('B-LASTNAME1', 7519)
('B-LASTNAME2', 1690)
('B-LASTNAME3', 539)
('B-PASS', 5055)
('B-PASSPORT', 8115)
('B-POSTCODE', 5752)
('B-SECADDRESS', 2639)
('B-SEX', 6674)
('B-SOCIALNUMBER', 8738)
('B-STATE', 5593)
('B-STREET', 5815)
('B-TEL', 6671)
('B-TIME', 12183)
('B-TITLE', 6668)
('B-USERNAME', 10144)
('I-BOD', 42930)
('I-BUILDING', 4413)
('I-CARDISSUER', 18)
('I-CITY', 29652)
('I-COUNTRY', 11535)
('I-DATE', 26820)
('I-DRIVERLICENSE', 53613)
('I-EMAIL', 84176)
('I-GEOCOORD', 3758)
('I-GIVENNAME1', 17272)
('I-GIVENNAME2', 4788)
('I-IDCARD', 40946)
('I-IP', 116577)
('I-LASTNAME1', 23556)
('I-LASTNAME2', 6088)
('I-LASTNAME3', 2138)
('I-PASS', 21866)
('I-PASSPORT', 32038)
('I-POSTCODE', 16

In [12]:
entity_classes = {tag[2:] for tag in bio_counter if tag.startswith('B-')}
print("unique entity classes:")
print(f"{len(entity_classes)=}")
for i in entity_classes:
    print(i)

unique entity classes:
len(entity_classes)=28
GIVENNAME2
GEOCOORD
BUILDING
POSTCODE
SECADDRESS
GIVENNAME1
CARDISSUER
COUNTRY
EMAIL
TEL
DRIVERLICENSE
BOD
SEX
PASSPORT
TITLE
LASTNAME3
CITY
SOCIALNUMBER
IDCARD
PASS
LASTNAME2
LASTNAME1
STREET
STATE
DATE
USERNAME
IP
TIME


In [13]:
print(Fore.CYAN + "least common labels:")
for i in reversed(bio_counter.most_common()[-5:]):
    print(Style.RESET_ALL + str(i))

least common labels:
('B-CARDISSUER', 5)
('I-CARDISSUER', 18)
('B-GEOCOORD', 535)
('B-LASTNAME3', 539)
('B-GIVENNAME2', 1589)


CARDISSUER only has 18 training examples which is effectively unlearnable.
this will probably cause near-zero F1 on this class unless i use upsampling and/or weighted loss.
For simplicity i will remove it

In [14]:
# remove all bio tags that appear less than 50 times (B and I CARDISSUER)
threshold = 50
# append keys to list to prevent:
# RuntimeError: dictionary changed size during iteration
keys_to_remove = []
for key, value in bio_counter.items():
    if value < threshold:
        keys_to_remove.append(key)
        
for key in keys_to_remove:
    del bio_counter[key]
    print(f"deleted tag {key}")
    
# print updated least common labels
print(Fore.CYAN + "updated least common labels:")
for i in reversed(bio_counter.most_common()[-5:]):
    print(Style.RESET_ALL + str(i))

deleted tag B-CARDISSUER
deleted tag I-CARDISSUER
updated least common labels:
('B-GEOCOORD', 535)
('B-LASTNAME3', 539)
('B-GIVENNAME2', 1589)
('B-LASTNAME2', 1690)
('I-LASTNAME3', 2138)


In [15]:
# remove cardissuer from dataset privacy mask
def remove_tag_from_pm(x, tag_names: list=["B-CARDISSUER", "I-CARDISSUER"]):
    pm: list[dict] = x["privacy_mask"]
    filtered = [d for d in pm if d["label"] not in tag_names]
    return {"privacy_mask": filtered}

ds.map(remove_tag_from_pm)

Map:   0%|          | 0/177677 [00:00<?, ? examples/s]

Map:   0%|          | 0/47728 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['source_text', 'target_text', 'privacy_mask', 'span_labels', 'mbert_text_tokens', 'mbert_bio_labels', 'id', 'language', 'set'],
        num_rows: 177677
    })
    validation: Dataset({
        features: ['source_text', 'target_text', 'privacy_mask', 'span_labels', 'mbert_text_tokens', 'mbert_bio_labels', 'id', 'language', 'set'],
        num_rows: 47728
    })
})

In [16]:
# Quick sanity check
sample = ds["train"][0]["privacy_mask"]
labels = [d["label"] for d in sample]
assert "B-CARDISSUER" not in labels and "I-CARDISSUER" not in labels
print(ds["train"].features)

{'source_text': Value('string'), 'target_text': Value('string'), 'privacy_mask': List({'value': Value('string'), 'start': Value('int64'), 'end': Value('int64'), 'label': Value('string')}), 'span_labels': Value('string'), 'mbert_text_tokens': List(Value('string')), 'mbert_bio_labels': List(Value('string')), 'id': Value('string'), 'language': Value('string'), 'set': Value('string')}


In [43]:
# understand bod, geocord, ip, pass, postcode
samples = []
labels = ["BOD", "GEOCOORD", "IP", "PASS", "POSTCODE", "STREET", "SECADDRESS"]
with tqdm(ds_train) as pbar:
    for row in pbar:
        if not labels:
            break
        for label in labels:
            if label in row["target_text"]:
                samples.append(row)
                labels.remove(label)

  0%|          | 61/29908 [00:00<00:07, 4092.40it/s]


In [44]:
print(labels)

[]


In [50]:
s = samples[5]
pprint([s["source_text"], s["privacy_mask"]])

['rgy, Guirard\n'
 '- Gender: Masculine\n'
 '- Username: kees.gyorgy02\n'
 '- Email: keesguirard@aol.com\n'
 '- Social Security Number: 464501286\n'
 '- ID Card Type: RGI\n'
 '- Telephone: 00758-30091\n'
 '- Nationality: Nederland\n'
 '- Address: 397, Kostverlorenkade\n'
 '- City: Amstelveen, State: NH\n'
 '- Postal Code: 1183 TM, Secondary Address: PB 73\n'
 '- IP Address: 11.47.34.34, Preferred Time: 2:58am\n'
 '\n'
 '**Applicant 3**\n'
 '- Last Name: Potkonjak\n'
 '\n'
 '**Applicant 4**\n'
 '- Last Name: Ucha, Mastrogiacomo, Raizner\n'
 '\n'
 '**Appl',
 [{'end': 12, 'label': 'LASTNAME2', 'start': 5, 'value': 'Guirard'},
  {'end': 32, 'label': 'SEX', 'start': 23, 'value': 'Masculine'},
  {'end': 58, 'label': 'USERNAME', 'start': 45, 'value': 'kees.gyorgy02'},
  {'end': 87, 'label': 'EMAIL', 'start': 68, 'value': 'keesguirard@aol.com'},
  {'end': 123, 'label': 'SOCIALNUMBER', 'start': 114, 'value': '464501286'},
  {'end': 143, 'label': 'IDCARD', 'start': 140, 'value': 'RGI'},
  {'end'

Bod is born on date? examples are dates
IP is ip address
pass is password
postcode is postalcode
geocoord is Geographic Coordinates ex [53.2, -1.3474] longitude and latitude

an address like "Address: 53, Zantelweg"
is marked: 
{'end': 282, 'label': 'BUILDING', 'start': 280, 'value': '53'},
{'end': 293, 'label': 'STREET', 'start': 284, 'value': 'Zantelweg'}

In [ ]:
data_dir = Path(os.getcwd()).parent / "data"
data_dir.mkdir(exist_ok=True)
print(data_dir)

/home/zelluzy/Desktop/pii_redaction_api/data


In [ ]:
sorted_bio_tags = ["O"] + sorted(tag for tag in bio_counter.keys() if tag != "O")

label2id = {label:i for i, label in enumerate(sorted_bio_tags)}
id2label = {i:label for i, label in enumerate(sorted_bio_tags)}
# validate info was parsed correctly
assert len(bio_counter.keys()) == len(sorted_bio_tags) == len(label2id) == len(id2label)

In [ ]:
# sort bio_tags with "O" as the first index
sorted_bio_tags = sorted(bio_counter)

# create dict to save collected data
label_info = {
    "bio_tags": sorted_bio_tags,
    "entity_classes": sorted(entity_classes),
    "label2id": label2id,
    "id2label": id2label,
    "train_bio_counts": dict(bio_counter),
    "description": {
        "bio_tags": "(list) all the labels the dataset was marked with. (B-, I-, O)",
        "entity_classes": "(list) all unique entities not including 'O' tag (eg. EMAIL)",
        "label2id": "(dict) labels mapped to their id (eg. 'O': 0 )",
        "id2label": "(dict) id mapped to label (eg. 0: 'O')",
        "train_bio_counts": "(dict) label mapped to their count in test dataset",
    }
}

In [ ]:
# remove all unnecessary columns before saving
columns_to_remove = [col for col in ds["train"].column_names if col not in ["source_text", "privacy_mask"]]
updated_ds = ds.remove_columns(columns_to_remove)
pprint(updated_ds["train"].features)

{'privacy_mask': List({'value': Value('string'), 'start': Value('int64'), 'end': Value('int64'), 'label': Value('string')}),
 'source_text': Value('string')}


In [ ]:
# save updated dataset
updated_ds.save_to_disk(data_dir/"updated_ai4privacy_300k_pii")

# save as json
with open(data_dir/"label_info.json", "w") as f:
    json.dump(label_info, f, indent=2)
    

Saving the dataset (0/1 shards):   0%|          | 0/177677 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/47728 [00:00<?, ? examples/s]